# Module 11: Interrupted Time Series Done Properly

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

An interrupted time series asks whether something changed when a policy took
effect. It is the most widely used design in public safety evaluation and the
easiest one to run badly.

This module does three things. It separates a **level change** from a **slope
change** and shows why that separation is harder than it looks. It runs the
**parallel trends** check that decides whether the design is usable at all.
And it is honest about what a single agency can and cannot establish.

**About 40 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

TREATED = ["A001", "A002", "A004", "A007", "A010"]

f = final.copy()
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]
f["treated"] = f["agency_id"].isin(TREATED).astype(int)
f["lo"] = np.log(f["n_arrests"])

pi = pd.PeriodIndex(f["year_month"], freq="M")
f["t"] = (pi.year.values - 2019) * 12 + pi.month.values - 1
f["mo"] = pi.month.values
f["yr"] = f["t"] / 12.0

START = (2023 - 2019) * 12 + 6        # 2023-07, the programme begins
FULL = (2023 - 2019) * 12 + 10        # 2023-11, fully in place

f["phase"] = ((f["t"] >= START) & (f["t"] < FULL)).astype(int)   # the transition months
f["post"] = (f["t"] >= FULL).astype(int)                         # the settled period
f["since"] = np.maximum(0, f["t"] - FULL + 1) / 12.0             # years since settled
f["sin1"] = np.sin(2 * np.pi * f["mo"] / 12)
f["cos1"] = np.cos(2 * np.pi * f["mo"] / 12)

pct = lambda b: 100 * (np.exp(b) - 1)


def poisson(formula, data):
    return smf.glm(formula, data, family=sm.families.Poisson(),
                   offset=data["lo"]).fit()


def show(fit, terms):
    for k in terms:
        lo, hi = fit.conf_int().loc[k]
        unit = " a year" if "since" in k else ""
        print(f"  {k:18s} {pct(fit.params[k]):+6.1f}%{unit}  "
              f"[{pct(lo):+6.1f}, {pct(hi):+6.1f}]")


clean = f[f["agency_id"] != "A007"]
print(f"{f['agency_id'].nunique()} agencies; "
      f"{int(f[f['agency_id'] == 'A001']['post'].sum())} settled months")

## 2. Three periods, not two

Almost every published interrupted time series splits the data at one date.
This programme had **three** periods: before, a four month phase in from July
2023, and the settled period from November.

Folding the phase in months into "before" says the programme had no effect
while it was being introduced. Folding them into "after" says it had its full
effect from day one. Both are wrong, and the fix is a third indicator.

## 3. A single agency, done correctly, and still not enough

In [ ]:
d = f[f["agency_id"] == "A001"]         # Stonewick

level = poisson("n_uof ~ yr + sin1 + cos1 + phase + post", d)
both = poisson("n_uof ~ yr + sin1 + cos1 + phase + post + since", d)

print("level change only")
show(level, ["post"])
print("\nlevel and slope change")
show(both, ["post", "since"])
print(f"\n  the truth: a 12.0 percent level drop, and no slope change at all")

Nothing here is wrong. The model has a trend, a season, a transition period and
a level term, and it returns an estimate whose interval is **22 percentage
points wide** and comfortably contains both zero and the truth.

**This is the correct result, not a failure.** One agency with 30 settled
months cannot resolve a 12 percent change. The reason is visible in the design
matrix.

In [ ]:
print(f"  correlation between the level indicator and the trend: "
      f"{np.corrcoef(d['yr'], d['post'])[0, 1]:.3f}")
print(f"  standard error on the level term {level.bse['post']:.4f}")
print(f"  standard error on the trend      {level.bse['yr']:.4f}")

pre = d[d["t"] < START]
print(f"\n  trend estimated from the pre period alone: "
      f"{pct(poisson('n_uof ~ yr + sin1 + cos1', pre).params['yr']):+.2f}% a year")
print(f"  trend estimated with the step in the model: "
      f"{pct(level.params['yr']):+.2f}% a year")
print(f"  the trend built into the data:             -4.88% a year")

The step sits in the last third of the series, so **the level indicator and the
trend are 0.82 correlated.** A downward step and a steeper slope explain the
same data, and the model splits the difference between them. That is why the
standard error on the level term is nearly five times the one on the trend.

This is not fixed by a better estimator. It is fixed by a comparison group,
which pins the trend down from agencies the programme did not touch.

## 4. Parallel trends: the check that decides everything

A comparison group is only useful if the treated agencies would have followed
it. That is not testable, but its observable implication is: **before the
programme, the two groups should have been moving at the same rate.**

In [ ]:
pre = f[f["t"] < START].copy()

for label, sub in [("A007 alone", pre[pre["agency_id"] == "A007"]),
                   ("the other treated", pre[(pre["treated"] == 1)
                                             & (pre["agency_id"] != "A007")]),
                   ("the comparison agencies", pre[pre["treated"] == 0])]:
    z = poisson("n_uof ~ yr", sub)
    lo, hi = z.conf_int().loc["yr"]
    print(f"  {label:24s} {pct(z.params['yr']):+6.2f}% a year  "
          f"[{pct(lo):+6.2f}, {pct(hi):+6.2f}]")

A007 was falling at twice everyone else's rate **four years before the training
existed**, and its interval does not come close to the comparison group's. It
began its own reform in 2019. Including it hands the programme credit for four
years of somebody else's work.

The formal version is an interaction between group and time over the pre
period.

In [ ]:
for label, sub in [("A007 excluded", pre[pre["agency_id"] != "A007"]),
                   ("A007 included", pre)]:
    z = poisson("n_uof ~ C(agency_id) + yr + treated:yr", sub)
    lo, hi = z.conf_int().loc["treated:yr"]
    print(f"  {label:16s} difference in pre trends "
          f"{pct(z.params['treated:yr']):+5.2f}% a year "
          f"[{pct(lo):+5.2f}, {pct(hi):+5.2f}]   p = {z.pvalues['treated:yr']:.3f}")

With A007 out, the pre trends differ by 0.70 percent a year and p is 0.60: the
design is usable.

With A007 in, the difference is 2.16 percent a year and **p is 0.0875**, which
would pass at the conventional threshold. Note that carefully. The group level
test only just notices a violation that is glaring when A007 is looked at on
its own. **A test that passes is not evidence that trends are parallel; with a
handful of agencies it is mostly evidence that the test is underpowered.**
Plot the pre period and look at every agency.

## 5. The controlled version

Now the same interrupted series design, with the comparison agencies in the
model and A007 out of it.

In [ ]:
both_c = poisson("n_uof ~ C(agency_id) + C(year_month) + treated:phase"
                 " + treated:post + treated:since", clean)
level_c = poisson("n_uof ~ C(agency_id) + C(year_month) + treated:phase"
                  " + treated:post", clean)

print("level and slope change")
show(both_c, ["treated:phase", "treated:post", "treated:since"])
print("\nlevel change only")
show(level_c, ["treated:phase", "treated:post"])
print(f"\n  AIC with the slope term {both_c.aic:.1f}, without it {level_c.aic:.1f}")

Read this slowly, because it is the point of the module.

The truth is a **pure 12 percent level drop with no slope change**. The level
only model returns 12.6 percent and is right. The model with a slope term
returns a level change of 3.4 percent and a continuing improvement of 7.6
percent a year, whose interval **excludes zero**.

**And AIC prefers the wrong model by 2.2 points.**

Anyone reporting the second specification would write that the programme
produced a modest immediate drop and continues to improve. That sentence is
entirely an artefact.

In [ ]:
s = np.sort(clean[clean["post"] == 1]["since"].unique())
path = both_c.params["treated:post"] + both_c.params["treated:since"] * s
print("  what the sloped model says the effect was, month by month:")
for i in range(0, len(s), 6):
    print(f"    {s[i]:.2f} years in   {pct(path[i]):+6.1f}%")
print(f"\n  average of that path over the settled window: {pct(path.mean()):+.1f}%")
print(f"  the level only estimate:                      "
      f"{pct(level_c.params['treated:post']):+.1f}%")
print(f"  the truth:                                    -12.0%")

The two specifications disagree at every single month and agree on the average.
**The average effect over a stated window is identified. The decomposition into
a level and a slope is not.**

The rule that follows: fit both, and if they disagree about the shape while
agreeing on the average, report the average and say that the series cannot
distinguish a step from a drift.

## 6. Autocorrelation robust standard errors

Interrupted time series residuals are often autocorrelated, which makes the
classical interval too narrow. The standard correction is Newey West.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

d2 = f[f["agency_id"] == "A001"].sort_values("t").copy()
d2["y"] = np.log(d2["n_uof"] / d2["n_arrests"])

ols = smf.ols("y ~ yr + sin1 + cos1 + phase + post", d2).fit()
hac = smf.ols("y ~ yr + sin1 + cos1 + phase + post", d2).fit(
    cov_type="HAC", cov_kwds={"maxlags": 12})

for name, z in [("classical", ols), ("Newey West, 12 lags", hac)]:
    lo, hi = z.conf_int().loc["post"]
    print(f"  {name:22s} {pct(z.params['post']):+6.1f}%  "
          f"[{pct(lo):+6.1f}, {pct(hi):+6.1f}]   se {z.bse['post']:.4f}")

p = acorr_ljungbox(ols.resid, lags=[12], return_df=True)["lb_pvalue"].iloc[0]
print(f"\n  Ljung Box on the residuals at lag 12: p = {p:.4f}")

The Newey West interval is **narrower**, not wider.

This surprises people, and the reason is in the Ljung Box p value of 0.27:
after the seasonal terms are in the model there is no autocorrelation left to
correct for, so the robust estimator is just a different finite sample
estimate of the same quantity, and here it happens to be smaller.

**Robust standard errors are not a safety margin.** Use them when the
diagnostic says you need them, and report the diagnostic either way. Applying
them reflexively neither guarantees a wider interval nor substitutes for
modelling the structure.

## Exercise

Two things could have been chosen differently: the date the intervention is
dated from, and how long the follow up runs. Vary both and see which one
actually changes the answer.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    for end_label, end in [("full follow up, through 2026-04", None),
                           ("short follow up, through 2024-06", "2024-06")]:
        g = f[f["agency_id"] != "A007"].copy()
        if end:
            g = g[g["year_month"] <= end]
        n_settled = int(((g["treated"] == 1) & (g["t"] >= FULL)).sum() / 4)
        print(f"  {end_label}   ({n_settled} settled months per agency)")
        for label, cut in [("dated from July 2023", START),
                           ("dated from November 2023", FULL)]:
            g["ind"] = ((g["treated"] == 1) & (g["t"] >= cut)).astype(float)
            z = poisson("n_uof ~ C(agency_id) + C(year_month) + ind", g)
            lo, hi = z.conf_int().loc["ind"]
            print(f"      {label:26s} {pct(z.params['ind']):+6.2f}%  "
                  f"[{pct(lo):+6.2f}, {pct(hi):+6.2f}]")
        print()
    print("  the truth is -12.0 percent once fully in place")
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

**The date barely matters.** Over the full follow up the two choices give 12.1
and 12.0 percent. The four phase in months are a small part of a 34 month
window, and the arithmetic dilution is a fifth of a percentage point.

**The length of the follow up matters enormously.** Cut the data off in June
2024, leaving eight settled months, and the same models give 7.2 and 5.0
percent with intervals that both include zero. A real 12 percent effect,
correctly specified, with the whole panel, is **undetectable eight months in**.

Three things follow.

The date should still be fixed before the estimates are seen. It happened not
to matter here, and you could not have known that in advance; an analyst who
tries both and reports the better one has run a hidden specification search
whatever the sizes turn out to be.

An evaluation that reports "no significant effect" after eight months has
mostly reported how long it waited. The honest version states the smallest
effect the available follow up could have detected, exactly as
[Module 9](Module_09_Rare_Events.ipynb) does for small agencies.

And the specification in section 5, with a separate phase in term, sidesteps
the date choice entirely. Use it whenever the transition is documented.

</details>

---

**Next:** [Module 12: Structural Breaks and Changepoints](Module_12_Structural_Breaks.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*